# RPA: clientes para o assistente de investimentos

Este notebook extrai os clientes da página do fork e envia a lista ao workflow do n8n. Execute o workflow em modo de teste antes de usar a URL de teste, ou ative-o para usar a URL de produção.

In [ ]:
%pip install -q requests beautifulsoup4

In [ ]:
import os
import requests
from bs4 import BeautifulSoup

URL_PAGINA = "https://leandrinhoaraujo.github.io/dio-lab-assistente-investimentos-rpa-n8n/"
# Opção 1: defina a variável N8N_WEBHOOK_URL.
# Opção 2: cole temporariamente a URL entre as aspas abaixo (não faça commit dela).
N8N_WEBHOOK_URL = os.getenv("N8N_WEBHOOK_URL", "")

resposta = requests.get(URL_PAGINA, timeout=30)
resposta.raise_for_status()
soup = BeautifulSoup(resposta.text, "html.parser")
linhas = soup.select("#clientes tbody tr")
if not linhas:
    raise ValueError("A tabela de clientes não foi encontrada. O GitHub Pages está ativo?")

clientes = []
for linha in linhas:
    colunas = linha.find_all("td")
    if len(colunas) != 4:
        raise ValueError("A tabela possui uma linha com formato inesperado.")
    clientes.append({
        "nome": colunas[0].get_text(strip=True),
        "email": colunas[1].get_text(strip=True),
        "saldo": colunas[2].get_text(strip=True),
        "perfil": colunas[3].get_text(strip=True),
    })

print(f"{len(clientes)} clientes encontrados.")
clientes

In [ ]:
if not N8N_WEBHOOK_URL:
    raise ValueError("Defina N8N_WEBHOOK_URL antes de enviar os dados.")

resposta_n8n = requests.post(
    N8N_WEBHOOK_URL,
    json={"clientes": clientes},
    timeout=60,
)
resposta_n8n.raise_for_status()
resultado = resposta_n8n.json()
print(f"{resultado['total']} recomendações geradas.")
for item in resultado["recomendacoes"]:
    print("-", item["mensagem"])